# Task 1 – Official-format pipeline and score estimate

This notebook walks through the final Task 1 system in `task1_agent/` and shows **why each choice was made**, using the **official CHIMERA-agent evaluation code** (not our own metric).

**What the official scoring rewards** (read from `evaluate.py`)
| Rule | Consequence for the design |
|---|---|
| Wrong biopsy decision → case score **0** | Decision accuracy matters twice (it gates the case score *and* drives F1) |
| Confidence: ordinal distance (clear/borderline/uncertain) | Predicting the typical expert level scores well; chance-corrected κ is *not* the metric |
| Variable weights: mean ordinal error over 10 variables | Same: typical-expert priors score well |
| Tool score = precision of revealed sections (extra reveals are penalised, missing ones are not) | Reveal only what the model really reads |
| Section grounding: a weighted variable needs its source section revealed | The model reads the MRI report → `pirads`, `psad`, `vol`, `cspca` are grounded |
| Output must match the schema, offline, no external APIs | Small local model, no LLM calls |

## 1. Setup
**Why:** we import the organisers' `evaluate.py` so every number below comes from their functions. The LLM rationale judge needs a local Ollama model, so it is switched off here: scores are the **deterministic part** of the case score (the official code re-normalises the weights when the judge is off).

In [1]:
import sys, os, json, subprocess, warnings
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
sys.path.insert(0, 'task1_agent')

DATA_DIR = '.'                                                        # features.csv + reasoning_audit.csv
EVAL_DIR = os.environ.get('CHIMERA_EVAL_DIR', 'CHIMERA-agent/evaluation')   # clone of github.com/DIAGNijmegen/CHIMERA-agent
import official_eval as OE, common as C
from featurize import featurize, FEATURES
ev = OE.load_official(EVAL_DIR)
print('official evaluator loaded; judge disabled:', not ev.USE_RATIONALE_JUDGE)

official evaluator loaded; judge disabled: True


## 2. Features come straight from the official JSON inputs
**Why:** at test time the container only receives `structured-prompt.json`, the clinical-data JSON and the neural-representation JSON. Every feature must therefore be computable from those. `featurize()` does this offline; here we check it against the Phase-1 columns and the official sample case.

Two improvements over Phase 1:
- **Lesion size** is now read as the largest dimension of "14 × 12 × 10 mm" (the old parser took the last number, so it was wrong). Gland dimensions are excluded.
- Features that **cannot be rebuilt reliably from the JSON** (the CSV-only `extracapsular_ext` and `lesion_zone` flags) are dropped.

**Model uses only the structured prompt + the MRI report.** Cross-validation showed that adding labs, PSA-trend, prior-notes and family-history features did *not* help (AUC 0.725 → 0.751 without them), and using fewer sections is exactly what the tool-efficiency score rewards. It also keeps the output honest: `reveal_sequence = ["radiology_report"]` is true.

In [2]:
sample_dir = os.environ.get('CHIMERA_SAMPLE_DIR', 'chimera-agent-baseline/input')
sp = json.load(open(f'{sample_dir}/structured-prompt.json'))
cd = json.load(open(f'{sample_dir}/prostate-biopsy-decision-clinical-data.json'))
print({k: (round(v, 3) if isinstance(v, float) else v) for k, v in featurize(sp, cd).items()})

{'age': 56.0, 'months': 3.0, 'cspca': 0.403, 'pirads': 4.0, 'n_comorbidities': 1.0, 'dre_abnormal': 1.0, 'psa_log': 1.841, 'psap_log': 1.308, 'psad_log': 0.095, 'psav_log': 1.182, 'vol_log': 3.892, 'lesion_mm': 14.0, 'lesion_found': 1.0, 'enc_followup': 0.0, 'enc_prior_pca': 1.0, 'enc_prior_neg': 0.0, 'bx_neg': 0.0, 'bx_none': 0.0, 'bx_pos': 1.0}


## 3. Train + honest score estimate
**Why these steps (all inside `train.py`)**
1. **Decision model:** equal-weight average of Logistic Regression, shallow Random Forest and shallow Gradient Boosting on 19 features. Different model families make different mistakes; shallow trees and strong regularisation suit 91 patients.
2. **Out-of-fold scores:** every patient is scored by a model that never saw them.
3. **Threshold:** chosen to maximise the *official* Task 1 score, smoothed over neighbouring thresholds so we do not chase noise.
4. **Reasoning policy** (what to write in `variable_weights`, `confidence`, `reveal_sequence`):
   - weights = the **typical expert level per variable**, adapted per case for `pirads`, `bx`, `psa` by small regularised models;
   - reveal only `radiology_report`;
   - confidence = typical expert level (`clear`) – the data showed no model-based rule beats it.
5. **Nested evaluation:** priors and heads are re-fitted inside each CV fold, then scored with `evaluate.py`.

In [3]:
r = subprocess.run([sys.executable, 'task1_agent/train.py', '--data', DATA_DIR, '--eval-dir', EVAL_DIR, '--out', 'task1_agent/model'],
                   capture_output=True, text=True)
print('\n'.join(l for l in r.stdout.splitlines() if 'arn' not in l))
if r.returncode: print(r.stderr[-1500:])

OOF AUC (pooled): 0.740
threshold (smoothed optimum): 0.48
priors + always-clear                Task1=0.656 mean_case=0.540 F1=0.773 acc=0.670 conf=0.787 varW=0.866 factorF1=0.748 tool=1.000 ground=0.625
priors + heads + always-clear        Task1=0.657 mean_case=0.542 F1=0.773 acc=0.670 conf=0.787 varW=0.879 factorF1=0.745 tool=1.000 ground=0.625
priors + heads + confidence rule     Task1=0.652 mean_case=0.531 F1=0.773 acc=0.670 conf=0.713 varW=0.879 factorF1=0.745 tool=1.000 ground=0.625
confidence mode chosen: always_clear
saved model to task1_agent/model


In [4]:
rep = json.load(open('task1_agent/model/training_report.json'))
tab = pd.DataFrame(rep['cv_estimates_official_deterministic']).T[['task1_score', 'mean_case_score', 'biopsy_F1', 'accuracy',
        'confidence_score', 'variable_weight_score', 'important_decisive_factor_score', 'tool_score', 'section_grounding_score']]
print(f"threshold = {rep['threshold']:.2f} | OOF AUC = {rep['oof_auc']:.3f} | confidence mode = {rep['conf_mode']}")
tab.round(3)

threshold = 0.48 | OOF AUC = 0.740 | confidence mode = always_clear


,task1_score,mean_case_score,biopsy_F1,accuracy,confidence_score,variable_weight_score,important_decisive_factor_score,tool_score,section_grounding_score
priors + always-clear,0.656,0.540,0.773,0.67,0.787,0.866,0.748,1.0,0.625
priors + heads + always-clear,0.657,0.542,0.773,0.67,0.787,0.879,0.745,1.0,0.625
priors + heads + confidence rule,0.652,0.531,0.773,0.67,0.713,0.879,0.745,1.0,0.625


**How to read this**
- `task1_score = (mean_case_score + biopsy_F1) / 2` – the official formula.
- `mean_case_score` is low-ish mainly because **about a third of decisions are wrong, and each wrong case scores 0**. Better decision accuracy is the biggest lever.
- `biopsy_F1` is computed for the "yes" class. About 62% of cases are "yes", so answering "yes" for everyone already gives F1 ≈ 0.76; the model's F1 gain is small.
- The score excludes the LLM-judged rationale component (20% of the case score on the real leaderboard), so treat it as an estimate with a plausible error of about ±0.05 on a hidden test set.

## 4. Why not the earlier reasoning layer?
We scored the **earlier** Task 1 output (model-derived importance, 3 sections, rule-based confidence) with the same official functions: **Task 1 ≈ 0.61** (mean case 0.455). The simple expert-prior policy above reaches about **0.66**. The reason is the metric: it measures *ordinal distance to the pathologist's levels* and *precision of revealed sections*, not chance-corrected agreement, so matching typical expert behaviour and revealing fewer sections wins.

**Trade-off you should know about:** constant-ish weights are not case-specific reasoning. They are justified as a *prediction of the expert annotation*, and we still adapt `pirads`, `bx`, `psa` and write a **case-specific free text**. The rationale judge explicitly penalises generic text, so the free text cites the real values of the case.

## 5. Run the real inference entry point on the official sample case
**Why:** this is exactly what runs on Grand Challenge: read `/input`, write the two files. We then check them with the official schema validator.

In [5]:
env = dict(os.environ, CHIMERA_INPUT=sample_dir, CHIMERA_OUTPUT='sample_output', CHIMERA_MODEL_DIR='task1_agent/model')
subprocess.run([sys.executable, 'task1_agent/inference.py'], env=env, check=True, capture_output=True)
d = json.load(open('sample_output/prostate-biopsy-decision.json')); rs = json.load(open('sample_output/prostate-biopsy-decision-reasoning.json'))
print('decision file :', json.dumps(d)); print(json.dumps(rs, indent=2))
pred = dict(rs); pred['biopsy_decision'] = d
print('official validate_record ->', ev.validate_record(pred, 'biopsy'))

decision file : "yes"
{
  "confidence": "clear",
  "variable_weights": {
    "psa": "important",
    "psad": "noted",
    "vol": "noted",
    "pirads": "important",
    "dre": "noted",
    "fh": "noted",
    "comorbidity": "noted",
    "cspca": "not_used",
    "age": "important",
    "bx": "important"
  },
  "reveal_sequence": [
    "radiology_report"
  ],
  "free_text": "Biopsy recommended (model risk score 0.62 vs decision threshold 0.48). Supporting this: PI-RADS 4 (14 mm lesion); DRE nodus. Context: PSA 5.3 ng/mL with rising velocity (+2.3); age 56. Based on the structured clinical variables and the MRI report."
}
official validate_record -> (True, 'ok')


## 6. Predictions for the unlabelled patients (official format)
**Why:** same code path as the container, run on the 104 patients without a label, so you can inspect decisions and texts. These cannot be scored.

In [6]:
import joblib
from predict import predict_case
bundle = joblib.load('task1_agent/model/model.joblib')
raw = pd.read_csv(f'{DATA_DIR}/features.csv')
from train import row_inputs
os.makedirs('task1_predictions_unlabelled', exist_ok=True); rows = []
for _, r in raw[raw.label.isna()].iterrows():
    s, c = row_inputs(r); dec, reas, p = predict_case(s, c, bundle)
    d = f"task1_predictions_unlabelled/{r.case_id}"; os.makedirs(d, exist_ok=True)
    json.dump(dec, open(f'{d}/prostate-biopsy-decision.json', 'w')); json.dump(reas, open(f'{d}/prostate-biopsy-decision-reasoning.json', 'w'), indent=2)
    rows.append({'case_id': r.case_id, 'risk': round(p, 3), 'decision': dec})
out = pd.DataFrame(rows); print(out.decision.value_counts().to_dict()); out.head()

{'yes': 83, 'no': 21}


,case_id,risk,decision
0,PT-pseudo_598f1d8f4e08,0.573,yes
1,PT-pseudo_5af00f9153c2,0.680,yes
2,PT-pseudo_5dc2eb70fbd7,0.406,no
3,PT-pseudo_5f263f325019,0.583,yes
4,PT-pseudo_62c93b0c3dba,0.822,yes


## 7. What is still needed before submitting
1. **Docker packaging** (offline): copy `task1_agent/` including `model/`, install scikit-learn/numpy/pandas/joblib, and set the entry point to `inference.py`. No LLM, RAG index or internet is needed.
2. **Check the filenames** your Grand Challenge algorithm interface expects (`prostate-biopsy-decision.json`, `prostate-biopsy-decision-reasoning.json`).
3. **Run the official evaluator** (`do_test_run.sh` with `USE_RATIONALE_JUDGE=1`, GPU + Ollama) to get the real, judge-inclusive score before reporting it. Reporting must use the official pipeline.
4. **Main upside left:** decision accuracy. Anything that raises the correct-decision rate raises both parts of the score.